### Tool Decorator - Practice & Exercises

In this notebook, you will practice using the simplified `@tool` decorator to define custom tools.

#### Part 1: Follow-Along Practice

##### Goal / Question
Define custom tools using `@tool` decorator with and without Pydantic schemas, bind them to an agent, and execute queries.

##### Sample Input
```python
"Reverse the string hello"
```

##### Sample Output
olleh

##### Plan / Steps
1. Annotate functions using the `@tool` decorator.
2. Define Pydantic models for functions requiring multiple inputs.
3. Bind Pydantic structures to `@tool(args_schema=...)`.
4. Initialize model using `init_chat_model("llama-3.3-70b-versatile", model_provider="groq")`.
5. Collect decorated functions inside a list and feed them into the agent executor.

In [ ]:
from dotenv import load_dotenv
from langchain import hub
from langchain.agents import AgentExecutor, create_tool_calling_agent
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model

load_dotenv()

@tool()
def greet_user(name: str) -> str:
    """Greets the user by name."""
    return f"Hello, {name}!"

class ReverseStringArgs(BaseModel):
    text: str = Field(description="Text to be reversed")

@tool(args_schema=ReverseStringArgs)
def reverse_string(text: str) -> str:
    """Reverses the given string."""
    return text[::-1]

class ConcatenateStringsArgs(BaseModel):
    a: str = Field(description="First string")
    b: str = Field(description="Second string")

@tool(args_schema=ConcatenateStringsArgs)
def concatenate_strings(a: str, b: str) -> str:
    """Concatenates two strings."""
    return a + b

tools = [greet_user, reverse_string, concatenate_strings]
llm = init_chat_model("llama-3.3-70b-versatile", model_provider="groq")
prompt = hub.pull("hwchase17/openai-tools-agent")
agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)
executor = AgentExecutor.from_agent_and_tools(agent=agent, tools=tools, verbose=True)
print(executor.invoke({"input": "Reverse the string hello"}))

#### Part 2: Practice Exercises

##### Exercise: Random Number Generator Tool

##### Goal / Question
Create a decorated tool `RandomGenerator` that generates a random integer between a user-defined `min_val` and `max_val`.

##### Sample Input
```python
{"input": "Generate a random number between 1 and 10"}
```

##### Sample Output
A random integer value (e.g. 7).

##### Plan / Steps
1. Formulate a Pydantic class `RandomGeneratorArgs` with inputs `min_val` and `max_val`.
2. Define the Python function returning a random number utilizing `random.randint`.
3. Annotate the function using `@tool(args_schema=RandomGeneratorArgs)`.
4. Append to tools list, rebuild agent, and test.

In [ ]:
# TODO: Write your implementation here

<details>
<summary>💡 Click to see solution</summary>

```python
import random
class RandomGeneratorArgs(BaseModel):
    min_val: int = Field(description="Minimum integer value")
    max_val: int = Field(description="Maximum integer value")

@tool(args_schema=RandomGeneratorArgs)
def random_num(min_val: int, max_val: int) -> str:
    """Generates a random integer within a range."""
    return str(random.randint(min_val, max_val))

extended_tools = tools + [random_num]
dec_agent = create_tool_calling_agent(llm=llm, tools=extended_tools, prompt=prompt)
dec_executor = AgentExecutor.from_agent_and_tools(agent=dec_agent, tools=extended_tools, verbose=True)
print(dec_executor.invoke({"input": "Generate a random number between 1 and 10"}))
```
</details>